# 01 - 이산 masking과 block attention

**학습 목표**: MinerU-Diffusion 식 (1), (4), (7)의 직관을 구현합니다.

**실행 방법**: Python 3/Jupyter에서 cell을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 라이브러리 `random`만 사용합니다.

실제 2.5B 모델이 아닌 toy reproduction입니다.

In [ ]:
import random
# 작은 list 행렬을 사용해 block attention의 허용·차단 규칙을 눈으로 확인합니다.

MASK = '[MASK]'

def forward_mask(tokens, t, seed=0):
    # q(x_t|x_0): 각 token을 확률 t로 MASK로 바꿉니다.
    rng = random.Random(seed)
    return [MASK if rng.random() < t else token for token in tokens]

def block_attention(length, block_size):
    # 같은 block은 양방향, 이전 block은 허용, future block은 차단합니다.
    matrix = []
    for i in range(length):
        bi = i // block_size
        row = []
        for j in range(length):
            bj = j // block_size
            row.append(1 if bi == bj or bj < bi else 0)
        matrix.append(row)
    return matrix

tokens = list('TABLEOCR')
print('t=0.5:', forward_mask(tokens, t=0.5, seed=4))
mask = block_attention(length=8, block_size=4)
for row in mask:
    print(''.join('#' if value else '.' for value in row))

assert mask[1][3] == 1       # 같은 첫 block의 미래 위치도 볼 수 있음
assert mask[1][4] == 0       # future block은 볼 수 없음
assert mask[6][1] == 1       # 이전 block은 볼 수 있음


`####....` 행들은 첫 block의 양방향 attention, `########` 행들은 두 번째 block이 과거와 현재를 모두 보는 구조입니다. vision/prompt token은 실제 모델에서 모든 block의 조건으로 추가됩니다.